In [114]:
import sys
sys.path.append('./Textual-Anomaly-Detection-Framework/Anomaly Detection Framework')

from Data_Preparation.Embedding import embedding_encoder
from Data_Preparation.Tac import tac
from Data_Preparation import utils
from Modelisation.FlowMatching import flow_matching
from Modelisation.Baselines.OCSVM import ocsvm
from Modelisation.Baselines.CVDD.utils import build_vocab, cvdd_model_pipeline
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net
from Modelisation.Baselines.RSRAE.model import CAE
from utils import save_results, create_tables

import torch
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader
import optuna
import torch
from torch import nn, Tensor
import numpy as np
from transformers import AutoTokenizer
from datasets import Dataset, concatenate_datasets
import time
import tensorflow as tf
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [115]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [116]:
train_20ng_, test_20ng_ = utils.import_dataset(name="20newsgroups", batch_size=BATCH_SIZE)
train_reuters_, test_reuters_ = utils.import_dataset(name="reuters", batch_size=BATCH_SIZE)
# train_wos = utils.import_dataset(name="WOS", batch_size=BATCH_SIZE)
# train_dbpedia14, test_dbpedia14 = utils.import_dataset(name="DBpedia14", batch_size=BATCH_SIZE)
train_agnews_, test_agnews_ = utils.import_dataset(name="agnews", batch_size=BATCH_SIZE)

20newsgroups dataset importing .... 




Repo card metadata block was not found. Setting CardData to empty.


reuters dataset importing .... 


agnews dataset importing .... 




In [117]:
train_agnews_ = utils.preprocess(train_agnews_.dataset)
test_agnews_ = utils.preprocess(test_agnews_.dataset)

train_reuters_ = utils.preprocess(train_reuters_.dataset)
test_reuters_ = utils.preprocess(test_reuters_.dataset)

train_20ng_ = utils.preprocess(train_20ng_.dataset)
test_20ng_ = utils.preprocess(test_20ng_.dataset)

In [118]:
def train_test_val_split(train, test, inlier_topic, dataset_name, type_tac, anomaly_rate, verbose=False):
    
    train_inlier, train_anomaly = tac.textual_anomaly_contamination(train, dataset_name, inlier_topic, type_tac, anomaly_rate, True)

    n_inliers_val = int(0.1 * len(train_inlier))
    inlier_indices = np.random.choice(len(train_inlier), n_inliers_val, replace=False)
    val_inlier_dataset = train_inlier.select(inlier_indices)

    train_inlier = train_inlier.select([i for i in range(len(train_inlier)) if i not in inlier_indices])
    
    n_anomalies_val = int(n_inliers_val / 0.9 * 0.1)
    anomaly_indices = np.random.choice(len(train_anomaly), n_anomalies_val, replace=False)
    val_anomaly_dataset = train_anomaly.select(anomaly_indices)

    val_ = concatenate_datasets([val_inlier_dataset, val_anomaly_dataset]).shuffle(seed=42)
    
    if verbose:
        print("TRAINSET")
        print(train_inlier)
        print(train_anomaly)
    
    if verbose:
        print("\nVALSET")
        print(val_)
        print()

    test_ = tac.textual_anomaly_contamination(test, dataset_name, inlier_topic, type_tac, anomaly_rate, False)
    if verbose:
        print("TESTSET")
        print(test_)

    return train_inlier, train_anomaly, val_, test_

In [100]:
inlier_topic = 'World'
dataset_name = 'agnews'
type_tac = 'fate' 
anomaly_rate = 0.1

train_inlier_agnews, train_anomaly_agnews, val_agnews, test_agnews = train_test_val_split(train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, True)

TRAINSET
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 27000
})
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 3333
})

VALSET
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 3333
})

TESTSET
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 2111
})


In [119]:
inlier_topic = 'acq'
dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters = train_test_val_split(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate, True)

TRAINSET
Dataset({
    features: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title', 'anomaly_class'],
    num_rows: 1437
})
Dataset({
    features: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title', 'anomaly_class'],
    num_rows: 177
})

VALSET
Dataset({
    features: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title', 'anomaly_class'],
    num_rows: 176
})

TESTSET
Dataset({
    features: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title', 'anomaly_class'],
    num_rows: 773
})


In [9]:
model_name = 'all-MiniLM-L6-v2'
sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

# model_name = 'distilbert-base-uncased'

# bertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'bert')

# model_name = 'glove_300d.kv'

# gloveEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'glove')

In [10]:
train_inlier_agnews = sentencebertEncoder.forward(train_inlier_agnews)
# train_anomaly_dl_20ng = sentencebertEncoder.forward(train_anomaly_dl_20ng)

test_agnews = sentencebertEncoder.forward(val_agnews)
# test_dl_20ng = sentencebertEncoder.forward(test_dl_20ng)

val_agnews = sentencebertEncoder.forward(test_agnews)
# val_20ng = sentencebertEncoder.forward(val_20ng

In [11]:
# X_inlier = Tensor(train_inlier_dl_20ng['sbert_embeddings']).to(device)
X_inlier = Tensor(train_inlier_agnews['sbert_embeddings']).to(device)
# X_inlier = Tensor(train_inlier_dl_20ng['glove_embedding']).to(device)

X_test =  Tensor(test_agnews['sbert_embeddings']).to(device)
y_test = np.array(test_agnews['anomaly_class'])

X_val =  Tensor(val_agnews['sbert_embeddings']).to(device)
y_val = np.array(val_agnews['anomaly_class'])

print(X_inlier.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

torch.Size([27000, 384])
torch.Size([3333, 384])
(3333,)
torch.Size([3333, 384])
(3333,)


## FM 

### Hyperparameters Search

In [205]:
batch_size_default = 32
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size_default, shuffle=True)
input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def objective(trial):

    # batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    # n_epochs = trial.suggest_int("n_epochs", 1000, 2000, step=100)
    n_epochs = trial.suggest_int("n_epochs", 50, 500, step=50)
    source = trial.suggest_categorical("source", ["sphere", "sphere-noised", "uniform"])
    # source = trial.suggest_categorical("source", ["gaussian", "sphere", "sphere-noised"])
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0, 1e-3, log=False)

    dl_train = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    flow_model = flow_matching.FlowMatching(source, X_inlier.cpu(), input_dim, latent_dim, sinu, device).to(device)

    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    print("\n############################################################")
    print(f"batch_size: {batch_size} | n_epochs:{n_epochs} | source: {source} | lr: {lr} | weight_decay: {weight_decay}\n")
    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)
    flow_model_trained = fm_trainer.train(dl_train, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    

    auc, fpr95, ap = fm_trainer.test(X_val, y_val, score_type='norm', solver_type='midpoint', n_steps=10)

    score = auc + ap - fpr95
    
    print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f} | SCORE: {score: .4f}")  
    
#     ocsvm_kwargs = {
#         "nu": 0.1,
#         "kernel": 'rbf',
#         "gamma": 'scale'
#         }
#     clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

#     _ = clf.predict(X_test.cpu().detach())           
#     scores_val = clf.decision_function(X_val.cpu().detach())

#     auc, ap, fpr95 = ev.evaluation(y_val, scores_val, verbose=False)
#     print(f"OCSVM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")
    
    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  

print("Best hyperparameters:", study.best_params)
print("Best score:", study.best_value)

[I 2025-11-28 16:03:15,165] A new study created in memory with name: no-name-7775831d-88fb-4ada-99ec-b6391437508e



############################################################
batch_size: 512 | n_epochs:250 | source: sphere | lr: 0.06761963353883615 | weight_decay: 0.0004891648540049783

 step 0 -> loss : 0.09610
 step 50 -> loss : 0.00497
 step 100 -> loss : 0.00494
 step 150 -> loss : 0.00502
 step 200 -> loss : 0.00499


[I 2025-11-28 16:04:36,252] Trial 0 finished with value: 0.6116652461493307 and parameters: {'batch_size': 512, 'n_epochs': 250, 'source': 'sphere', 'lr': 0.06761963353883615, 'weight_decay': 0.0004891648540049783}. Best is trial 0 with value: 0.6116652461493307.


AUC: 0.8437 | FPR@95: 0.6620 | AP: 0.4300
FM --> AUC: 0.8437 | FPR@95: 0.6620 | AP: 0.4300 | SCORE:  0.6117

############################################################
batch_size: 256 | n_epochs:250 | source: sphere-noised | lr: 0.003934403626874193 | weight_decay: 0.0001532196096424735

 step 0 -> loss : 0.06685
 step 50 -> loss : 0.06747
 step 100 -> loss : 0.06778
 step 150 -> loss : 0.06760
 step 200 -> loss : 0.06663


[I 2025-11-28 16:06:40,272] Trial 1 finished with value: 0.6215368668294581 and parameters: {'batch_size': 256, 'n_epochs': 250, 'source': 'sphere-noised', 'lr': 0.003934403626874193, 'weight_decay': 0.0001532196096424735}. Best is trial 1 with value: 0.6215368668294581.


...

AUC: 0.8373 | FPR@95: 0.5767 | AP: 0.4174
FM --> AUC: 0.8373 | FPR@95: 0.5767 | AP: 0.4174 | SCORE:  0.6781

############################################################
batch_size: 1024 | n_epochs:350 | source: sphere | lr: 0.012815755218821105 | weight_decay: 0.0006788611434387134

 step 0 -> loss : 0.00498
 step 70 -> loss : 0.00496
 step 140 -> loss : 0.00496
 step 210 -> loss : 0.00494
 step 280 -> loss : 0.00495


[I 2025-11-28 16:15:33,303] Trial 8 finished with value: 0.6925191942336256 and parameters: {'batch_size': 1024, 'n_epochs': 350, 'source': 'sphere', 'lr': 0.012815755218821105, 'weight_decay': 0.0006788611434387134}. Best is trial 3 with value: 0.7399300505928488.


AUC: 0.8476 | FPR@95: 0.5717 | AP: 0.4166
FM --> AUC: 0.8476 | FPR@95: 0.5717 | AP: 0.4166 | SCORE:  0.6925

############################################################
batch_size: 256 | n_epochs:50 | source: sphere | lr: 0.04120620965485323 | weight_decay: 6.100001078133932e-05

 step 0 -> loss : 0.00625
 step 10 -> loss : 0.00494
 step 20 -> loss : 0.00490
 step 30 -> loss : 0.00493
 step 40 -> loss : 0.00493


[I 2025-11-28 16:15:53,545] Trial 9 finished with value: 0.6463586290616398 and parameters: {'batch_size': 256, 'n_epochs': 50, 'source': 'sphere', 'lr': 0.04120620965485323, 'weight_decay': 6.100001078133932e-05}. Best is trial 3 with value: 0.7399300505928488.


AUC: 0.8405 | FPR@95: 0.5933 | AP: 0.3992
FM --> AUC: 0.8405 | FPR@95: 0.5933 | AP: 0.3992 | SCORE:  0.6464
Best hyperparameters: {'batch_size': 1024, 'n_epochs': 150, 'source': 'uniform', 'lr': 0.02234147805474459, 'weight_decay': 0.0006524274414318924}
Best score: 0.7399300505928488


### Running

In [89]:
inlier_topic = 'religion'
dataset_name = '20newsgroups'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_agnews, _, _, _ = train_test_val_split(train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)

model_name = 'all-MiniLM-L6-v2'
sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

train_inlier_agnews_emb = sentencebertEncoder.forward(train_inlier_agnews)
X_inlier = Tensor(train_inlier_agnews_emb['sbert_embeddings']).to(device)


list_auc_fm = []
list_fpr_fm = []
list_ap_fm = []
list_auc_ocsvm = []
list_fpr_ocsvm = []
list_ap_ocsvm = []
list_auc_rsrae = []
list_fpr_rsrae = []
list_ap_rsrae = []

for i in range(5):
    
    print("\n##################################")
    print(f"Loading Dataset for the run {i+1}")
    
    inlier_topic = 'religion'
    dataset_name = '20newsgroups'
    type_tac = 'ruff' 
    anomaly_rate = 0.1

    _, _, _, test_agnews = train_test_val_split(train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)
    
    # print("A sample for valset : ")
    # print(val_20ng[0]['text'])
    # print()
    # print("\nVALSET")
    # print(val_20ng.num_rows)
    # print()
    print("A sample for testset : ")
    print(test_agnews[-1]['text'][:50])
    print()
    # print("TESTSET")
    # print(test_20ng.num_rows)

    

    test_agnews_emb = sentencebertEncoder.forward(test_agnews)
    # val_reuters_emb = sentencebertEncoder.forward(val_reuters)

    # print(X_inlier.shape)

    X_test =  Tensor(test_agnews_emb['sbert_embeddings']).to(device)
    y_test = np.array(test_agnews_emb['anomaly_class'])
    # print(X_test.shape, y_test.shape)
    
    # X_val =  Tensor(val_reuters_emb['sbert_embeddings']).to(device)
    # y_val = np.array(val_reuters_emb['anomaly_class'])
    # print(X_val.shape, y_val.shape)
    
    #########################################
    ################# OCSVM #################
    #########################################  
    
#     taac = time.time()
    
#     ocsvm_kwargs = {
#         "nu": 0.1,
#         "kernel": 'rbf',
#         "gamma": 'scale'
#         }
#     clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu(), ocsvm_kwargs)
    
#     tiic = time.time()
    
#     print(f"OCSVM finishing... after {(tiic-taac)/60:.3f} mn")

#     _ = clf.predict(X_test.cpu().detach())           
#     scores_test = clf.decision_function(X_test.cpu())

#     auc_ocsvm, ap_ocsvm, fpr95_ocsvm = ev.evaluation(y_test, scores_test, verbose=False)
#     print(f"OCSVM --> AUC: {auc_ocsvm:.4f} | FPR@95: {fpr95_ocsvm:.4f} | AP: {ap_ocsvm:.4f}\n")
    
#     list_auc_ocsvm.append(auc_ocsvm)
#     list_fpr_ocsvm.append(fpr95_ocsvm)    
#     list_ap_ocsvm.append(ap_ocsvm)  
    
    
    
    #########################################
    ################# RSRAE #################
    #########################################  
    
    cae = fit_rsrae(X_inlier)
    auc_rsrae, fpr95_rsrae, ap_rsrae = test_rsrae(X_test, y_test)
    
    list_auc_rsrae.append(auc_rsrae)
    list_fpr_rsrae.append(fpr95_rsrae)    
    list_ap_rsrae.append(ap_rsrae)  
    
    
    #################################################
    ################# FLOW MATCHING #################
    #################################################
    
#     # batch_size = 64
#     batch_size = study.best_params['batch_size']
#     X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

#     input_dim = X_inlier.shape[1]
#     latent_dim = 256
#     sinu = False
#     # lr = 1e-4
#     lr = study.best_params['lr']
#     # weight_decay = 1e-5
#     weight_decay = study.best_params['weight_decay']
#     # n_epochs = 1000
#     n_epochs = study.best_params['n_epochs']


#     target = X_inlier.cpu()
#     # source = 'sphere-noised'
#     source = study.best_params['source']


#     flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
#     optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
#     loss_fn = nn.MSELoss()

#     fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)
    
#     taac = time.time()

#     flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    
#     tiic = time.time()
    
#     print(f"\nFM finishing... after {(tiic-taac)/60:.3f} mn")

#     auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)
    
#     list_auc_fm.append(auc)
#     list_fpr_fm.append(fpr95)    
#     list_ap_fm.append(ap)    


##################################
Loading Dataset for the run 1
A sample for testset : 
what jetting do you recommend for a zx with standa



I0000 00:00:1764592834.738176 3513939 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78745 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:0b:00.0, compute capability: 8.0


 epoch 20/200 : loss = 137.183
 epoch 40/200 : loss = 117.602
 epoch 60/200 : loss = 101.479
 epoch 80/200 : loss = 88.043
 epoch 100/200 : loss = 76.743
 epoch 120/200 : loss = 67.163
 epoch 140/200 : loss = 58.973
 epoch 160/200 : loss = 51.940
 epoch 180/200 : loss = 45.859
 epoch 200/200 : loss = 40.584
AUC: 0.9049 | FPR@95: 0.5240 | AP: 0.6277

##################################
Loading Dataset for the run 2
A sample for testset : 
while i can see why they want to cut down on the t



I0000 00:00:1764592847.661634 3513939 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78745 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:0b:00.0, compute capability: 8.0


 epoch 20/200 : loss = 135.283
 epoch 40/200 : loss = 115.514
 epoch 60/200 : loss = 99.315
 epoch 80/200 : loss = 85.869
 epoch 100/200 : loss = 74.592
 epoch 120/200 : loss = 65.070
 epoch 140/200 : loss = 56.959
 epoch 160/200 : loss = 50.013
 epoch 180/200 : loss = 44.028
 epoch 200/200 : loss = 38.847
AUC: 0.9096 | FPR@95: 0.4292 | AP: 0.6448

##################################
Loading Dataset for the run 3
A sample for testset : 
from article ardieuxcsouiucedu by ardieuxcsouiuced



...

 epoch 20/200 : loss = 145.506
 epoch 40/200 : loss = 124.991
 epoch 60/200 : loss = 108.081
 epoch 80/200 : loss = 93.974
 epoch 100/200 : loss = 82.061
 epoch 120/200 : loss = 71.941
 epoch 140/200 : loss = 63.266
 epoch 160/200 : loss = 55.796
 epoch 180/200 : loss = 49.329
 epoch 200/200 : loss = 43.708
AUC: 0.9365 | FPR@95: 0.2854 | AP: 0.6726

##################################
Loading Dataset for the run 4
A sample for testset : 
i must say that i have been a customer of midwest 



I0000 00:00:1764592873.426902 3513939 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78745 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:0b:00.0, compute capability: 8.0


 epoch 20/200 : loss = 131.068
 epoch 40/200 : loss = 112.029
 epoch 60/200 : loss = 96.378
 epoch 80/200 : loss = 83.378
 epoch 100/200 : loss = 72.459
 epoch 120/200 : loss = 63.226
 epoch 140/200 : loss = 55.349
 epoch 160/200 : loss = 48.602
 epoch 180/200 : loss = 42.791
 epoch 200/200 : loss = 37.754
AUC: 0.9074 | FPR@95: 0.4366 | AP: 0.6093

##################################
Loading Dataset for the run 5
A sample for testset : 
actually i was hoping for barry bonds oh well firs



I0000 00:00:1764592886.298795 3513939 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78745 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:0b:00.0, compute capability: 8.0


 epoch 20/200 : loss = 141.291
 epoch 40/200 : loss = 121.348
 epoch 60/200 : loss = 104.874
 epoch 80/200 : loss = 91.112
 epoch 100/200 : loss = 79.527
 epoch 120/200 : loss = 69.668
 epoch 140/200 : loss = 61.246
 epoch 160/200 : loss = 53.986
 epoch 180/200 : loss = 47.703
 epoch 200/200 : loss = 42.245
AUC: 0.9273 | FPR@95: 0.3024 | AP: 0.6797


In [207]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="flow-matching",
    auc_mean=np.mean(list_auc_fm),
    ap_mean=np.mean(list_ap_fm),
    fpr_mean=np.mean(list_fpr_fm),
    auc_std = np.std(list_auc_fm),
    ap_std =  np.std(list_ap_fm),
    fpr_std = np.std(list_fpr_fm) 
)

Nouveaux résultats ajoutés pour (agnews, Sci/Tech, sentence_bert, flow-matching).


In [208]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="ocsvm",
    auc_mean=np.mean(list_auc_ocsvm),
    ap_mean=np.mean(list_ap_ocsvm),
    fpr_mean=np.mean(list_fpr_ocsvm),
    auc_std = np.std(list_auc_ocsvm),
    ap_std =  np.std(list_ap_ocsvm),
    fpr_std = np.std(list_fpr_ocsvm) 
)

Nouveaux résultats ajoutés pour (agnews, Sci/Tech, sentence_bert, ocsvm).


In [90]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="RSRAE",
    auc_mean=np.mean(list_auc_rsrae),
    ap_mean=np.mean(list_ap_rsrae),
    fpr_mean=np.mean(list_fpr_rsrae),
    auc_std = np.std(list_auc_rsrae),
    ap_std =  np.std(list_ap_rsrae),
    fpr_std = np.std(list_fpr_rsrae) 
)

Nouveaux résultats ajoutés pour (20newsgroups, religion, sentence_bert, RSRAE).


In [113]:
create_tables()

fichier généré : /home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Results/tables.tex


In [195]:
batch_size = 1024
# batch_size = study.best_params['batch_size']
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
lr = 1e-3
# lr = study.best_params['lr']
weight_decay = 0
# weight_decay = study.best_params['weight_decay']
n_epochs = 1500
# n_epochs = study.best_params['n_epochs']

target = X_inlier.cpu()
source = 'gaussian'
# source = study.best_params['source']

flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

 step 0 -> loss : 0.85965
 step 300 -> loss : 0.47459
 step 600 -> loss : 0.46203
 step 900 -> loss : 0.47441
 step 1200 -> loss : 0.46933
AUC: 0.8695 | FPR@95: 0.4947 | AP: 0.4348


## Baselines

### OCSVM

In [171]:
ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

_ = clf.predict(X_test.cpu().detach())           
scores_test = clf.decision_function(X_test.cpu().detach())

auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.8973 | FPR@95: 0.5157 | AP: 0.6541


In [95]:
indi = np.argsort(scores_test)[::-1]
print(scores_test[indi])
print(y_test[indi][:50])

[ 26.44050024  23.55905367  23.29357872 ... -33.74652881 -33.90646543
 -35.42626103]
[1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 0 1 1 1 1 1]


### CVDD

In [14]:
type_emb = 'glove'
emb_model = 'distilbert-base-uncased'
attention_size = 150
n_attention_heads = 10
lr = 0.01
lr_milestones = (40, 60)
n_epochs = 100
lambda_p = 1.0
alpha_scheduler = 'logarithmic'

In [15]:
if type_emb == 'bert':
    tokenizer = AutoTokenizer.from_pretrained(emb_model)
    vocab = None

elif type_emb in ('glove', 'fasttext'):
    corpus = train_inlier_dl_20ng['text']
    vocab = build_vocab(corpus,min_freq=1)
    tokenizer = None

In [16]:
cvdd_model, dl_train, dl_test = cvdd_model_pipeline(train_inlier_dl_20ng, test_dl_20ng, attention_size, n_attention_heads, 
                                               type_emb, 500, 64, True, device, tokenizer, vocab)

In [17]:
cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=lr_milestones,
                                    n_epochs=n_epochs, lambda_p=lambda_p,
                                    alpha_scheduler=alpha_scheduler, weight_decay=1e-4, device=device)

model_trained = cvdd_trainer.train(cvdd_model, dl_train)

Starting training...
KMean starts
KMeans finish
| Epoch: 001/010 | Train Time: 0.517s | Train Loss: 0.139542 |
| Epoch: 002/010 | Train Time: 0.444s | Train Loss: 0.056700 |
| Epoch: 003/010 | Train Time: 0.441s | Train Loss: 0.049708 |
| Epoch: 004/010 | Train Time: 0.439s | Train Loss: 0.046049 |
| Epoch: 005/010 | Train Time: 0.441s | Train Loss: 0.042732 |
| Epoch: 006/010 | Train Time: 0.439s | Train Loss: 0.041651 |
| Epoch: 007/010 | Train Time: 0.439s | Train Loss: 0.040930 |
| Epoch: 008/010 | Train Time: 0.439s | Train Loss: 0.040041 |
| Epoch: 009/010 | Train Time: 0.439s | Train Loss: 0.039830 |
| Epoch: 010/010 | Train Time: 0.441s | Train Loss: 0.039731 |
Training Time: 5.272s
Finished training. 



In [18]:
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.5008 | FPR@95: 0.9654 | AP: 0.1162


### RSRAE

In [38]:
def fit_rsrae(X_inlier):
    
    input_shape = (X_inlier.shape[1],)
    dim_latent = 10
    activation = tf.nn.relu
    loss = 'MSE'
    if_rsr = True
    if_enforce_proj = False
    if_all_alt = False
    lambda1 = 0.0025
    renormalize = False
    bn = True
    
    cae = CAE(input_shape=input_shape, hidden_layer_sizes=(32,64,128), intrinsic_size=dim_latent,
                          activation=activation,
                          norm_type='L21', loss_norm_type=loss,
                          if_rsr=if_rsr, enforce_proj=if_enforce_proj, all_alt=if_all_alt,
                          learning_rate=0.00025, 
                          epoch_size=200, batch_show=20, 
                          lambda1=lambda1,
                          normalize=renormalize,
                          bn=bn,
                          random_seed=None)
    
    cae.fit(X_inlier.cpu().numpy(), X_inlier.cpu().numpy())    
    
    return cae

In [39]:
cae = fit_rsrae(X_inlier)

I0000 00:00:1764585944.333535 3513939 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78745 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:0b:00.0, compute capability: 8.0


 epoch 20/200 : loss = 13.709
 epoch 40/200 : loss = 1.831
 epoch 60/200 : loss = 0.863
 epoch 80/200 : loss = 0.809
 epoch 100/200 : loss = 0.805
 epoch 120/200 : loss = 0.803
 epoch 140/200 : loss = 0.801
 epoch 160/200 : loss = 0.798
 epoch 180/200 : loss = 0.796
 epoch 200/200 : loss = 0.795


In [40]:
def test_rsrae(X_test, y_test):
    features = cae.get_output(X_test.cpu().numpy())
    flat_output = np.reshape(features, (np.shape(X_test.cpu().numpy())[0], -1))
    flat_input = np.reshape(X_test.cpu().numpy(), (np.shape(X_test.cpu().numpy())[0], -1))
    
    cosine_similarity = np.sum(flat_output * flat_input, -1) / (np.linalg.norm(flat_output, axis=-1) + 0.000001) / (np.linalg.norm(flat_input, axis=-1) + 0.000001)

    auc = roc_auc_score(y_test, -cosine_similarity)
    ap = average_precision_score(y_test, -cosine_similarity)
    fpr, tpr, thresholds = roc_curve(y_test, -cosine_similarity)
    idx = np.where(tpr >= 0.95)[0][0]
    fpr95 = fpr[idx]

    print(f"RSRAE --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  
    
    return auc, fpr95, ap

In [41]:
auc, fpr95, ap = test_rsrae(X_test, y_test)

AUC: 0.8988 | FPR@95: 0.3380 | AP: 0.4747
